# 04 — Base Detection

## What Is a Base?
A **base** is a tight consolidation zone — one or more consecutive candles where price idles rather than trending.  
Institutionally it's where large orders are stacked before a directional move.

## Per-Candle Gate (already computed in NB02)
A candle is *base-like* when its body covers at most 50 % of its total range:

$$\text{body\_to\_range\_ratio} = \frac{|\text{close} - \text{open}|}{\text{high} - \text{low}} \leq 0.50$$

NB04 does **not** re-filter individual candles; it only clusters candles already flagged `is_base = True`.

## Cluster-Level Gates
Once a contiguous run is found, the cluster as a whole must pass two tightness checks:

| Gate | Formula | Threshold | Meaning |
|------|---------|-----------|--------|
| Length | $1 \leq \text{count} \leq 5$ | 1 – 5 | Not a single spike, not a long chop |
| Width / ATR | $\dfrac{\text{base\_high} - \text{base\_low}}{\overline{\text{ATR}}} \leq 2.5$ | 2.5 | Cluster height is narrow relative to volatility |
| Compactness | $\dfrac{\text{close\_max} - \text{close\_min}}{\text{base\_width}} \leq 0.80$ | 0.80 | Closing prices cluster together (not scattered top-to-bottom) |

## What's Built Here
- `utils/base_detector.py` — `find_base_clusters()`, `evaluate_cluster()`, `detect_bases()`
- This notebook runs those functions on all four timeframes and visualises the result

## 1. Setup & load data

In [5]:
import sys; sys.path.append("..")

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from utils.data_loader import load_all_timeframes
from utils.models import CandlePrimitives, add_atr
from utils.base_detector import detect_bases
from utils.config import (
    BASE_MIN_CANDLES, BASE_MAX_CANDLES, BASE_MAX_ATR_WIDTH, BASE_COMPACTNESS_MAX,
    COLOR_BULL, COLOR_BEAR,
    CHART_BG as BG, CHART_GRID as GRID,
)

pio.renderers.default = "notebook_connected"

SYMBOL = "USDJPY=X"
raw  = load_all_timeframes(SYMBOL, align=True)
data = {tf: add_atr(CandlePrimitives.enrich_dataframe(df)) for tf, df in raw.items()}

print(f"Loaded {len(data)} timeframes: {list(data.keys())}")
for tf, df in data.items():
    print(f"  {tf:>4}: {len(df):>5} candles  |  is_base = {df['is_base'].sum()}")

Loaded 4 timeframes: ['1wk', '1d', '4h', '1h']
   1wk:   104 candles  |  is_base = 63
    1d:   581 candles  |  is_base = 308
    4h:  3174 candles  |  is_base = 1854
    1h: 12259 candles  |  is_base = 7215


## 2. Detect base clusters

**`find_base_clusters(df)`** scans left-to-right:  
- Open a cluster when `is_base == True`  
- Extend while the next candle is also base *and* count < `BASE_MAX_CANDLES`  
- Save and resume after the cluster  

**`evaluate_cluster(df, cluster)`** then measures two ratios and applies the three gates.

**`detect_bases(df)`** wraps both into a single call → `(passed, failed)`.

In [6]:
results = {}
for tf, df in data.items():
    passed, failed = detect_bases(df)
    results[tf] = {"passed": passed, "failed": failed}

# Summary table
rows = []
for tf, r in results.items():
    total = len(r["passed"]) + len(r["failed"])
    rows.append({
        "timeframe":      tf,
        "base_candles":   int(data[tf]["is_base"].sum()),
        "clusters_found": total,
        "passed":         len(r["passed"]),
        "failed":         len(r["failed"]),
        "pass_rate":      f"{100*len(r['passed'])/total:.0f}%" if total else "—",
    })

pd.DataFrame(rows).set_index("timeframe")

,base_candles,clusters_found,passed,failed,pass_rate
timeframe,,,,,
1wk,63,27,25,2,93%
1d,308,160,159,1,99%
4h,1854,844,811,33,96%
1h,7215,3243,3107,136,96%


## 3. Inspect — daily clusters

In [7]:
passed_1d = results["1d"]["passed"]
failed_1d = results["1d"]["failed"]
print(f"1d  →  {len(passed_1d)} passed,  {len(failed_1d)} failed\n")

all_1d = sorted(passed_1d + failed_1d, key=lambda c: c["start"])
(
    pd.DataFrame(all_1d)
    [["start", "end", "count",
      "base_high", "base_low", "base_width", "avg_atr",
      "width_atr_ratio", "compactness_ratio",
      "width_passed", "compactness_passed", "passed"]]
)

1d  →  159 passed,  1 failed



,start,end,count,base_high,base_low,base_width,avg_atr,width_atr_ratio,compactness_ratio,width_passed,compactness_passed,passed
0,0,0,1,156.44501,155.34900,1.09601,1.09601,1.000,0.000,True,True,True
1,3,7,5,158.25700,155.71001,2.54700,1.04281,2.442,0.207,True,True,True
2,8,11,4,158.22900,157.14999,1.07901,0.98519,1.095,0.375,True,True,True
3,15,16,2,159.93401,158.74001,1.19400,0.92159,1.296,0.005,True,True,True
4,18,19,2,161.28200,160.25800,1.02400,0.90685,1.129,0.129,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
155,559,561,3,158.44000,156.70700,1.73300,1.02849,1.685,0.414,True,True,True
156,564,567,4,159.35001,158.57899,0.77101,0.86451,0.892,0.236,True,True,True
157,570,570,1,159.04201,158.75000,0.29201,0.72465,0.403,0.000,True,True,True
158,574,575,2,159.39999,159.09000,0.31000,0.63052,0.492,0.342,True,True,True


## 4. Visualize — all 4 timeframes

Each chart has:
- **Top panel** — candlestick with cluster overlays  
  - Green shading + `✓Nc` label → passed all gates  
  - Red dotted → failed at least one gate  
- **Bottom panel** — ATR(14) area, so the width/ATR ratio is visually intuitive

In [8]:
PLOT_MONTHS = 2  # only render this many months in charts (detection still uses full history)

def plot_base_detection(
    df: pd.DataFrame,
    passed: list[dict],
    failed: list[dict],
    tf: str,
) -> go.Figure:
    # ── slice to last PLOT_MONTHS for performance ────────────────────────────
    cutoff   = df.index[-1] - pd.DateOffset(months=PLOT_MONTHS)
    df_plot  = df[df.index >= cutoff].copy()
    offset   = len(df) - len(df_plot)   # how many bars were trimmed from the front

    # remap cluster iloc positions to the sliced df; drop clusters entirely before window
    def _remap(c: dict) -> dict:
        return {**c, "start": max(c["start"] - offset, 0), "end": c["end"] - offset}

    passed_plot = [_remap(c) for c in passed if c["end"] >= offset]
    failed_plot = [_remap(c) for c in failed if c["end"] >= offset]

    # ── build figure ─────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.75, 0.25],
        vertical_spacing=0.03,
    )

    fig.add_trace(go.Candlestick(
        x=df_plot.index,
        open=df_plot["open"], high=df_plot["high"],
        low=df_plot["low"],   close=df_plot["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name="Price",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=df_plot.index, y=df_plot["atr"],
        mode="lines",
        line=dict(color="#7c83fd", width=1.5),
        fill="tozeroy",
        fillcolor="rgba(124,131,253,0.10)",
        name="ATR(14)",
        hovertemplate="%{x}<br>ATR: %{y:.5f}<extra></extra>",
    ), row=2, col=1)

    for cluster in passed_plot:
        x0, x1 = df_plot.index[cluster["start"]], df_plot.index[cluster["end"]]
        fig.add_vrect(
            x0=x0, x1=x1,
            fillcolor="rgba(38,166,154,0.18)",
            line=dict(color="#26a69a", width=1, dash="dot"),
            row=1, col=1,
            annotation_text=f"✓{cluster['count']}c",
            annotation_font=dict(size=9, color="#26a69a"),
            annotation_position="top left",
        )

    for cluster in failed_plot:
        x0, x1 = df_plot.index[cluster["start"]], df_plot.index[cluster["end"]]
        fig.add_vrect(
            x0=x0, x1=x1,
            fillcolor="rgba(239,83,80,0.08)",
            line=dict(color="#ef5350", width=1, dash="dot"),
            row=1, col=1,
        )

    fig.update_layout(
        title=(
            f"Base Detection — {SYMBOL} {tf} (last {PLOT_MONTHS}m)"
            f"  |  ✓ {len(passed_plot)} valid  ·  ✗ {len(failed_plot)} rejected"
            f"  (full history: ✓{len(passed)} / ✗{len(failed)})"
        ),
        height=620,
        plot_bgcolor=BG, paper_bgcolor=BG,
        font_color="white",
        xaxis_rangeslider_visible=False,
        hovermode="x unified",
        legend=dict(orientation="h", y=1.04),
    )
    shared_axis = dict(gridcolor=GRID, showgrid=True, zeroline=False, linecolor=GRID)
    fig.update_xaxes(**shared_axis)
    fig.update_yaxes(**shared_axis)
    fig.update_yaxes(title_text="Price",   row=1, col=1)
    fig.update_yaxes(title_text="ATR(14)", row=2, col=1)
    fig.update_xaxes(showticklabels=True,  row=2, col=1)
    return fig


for tf in ["1wk", "1d", "4h", "1h"]:
    fig = plot_base_detection(data[tf], results[tf]["passed"], results[tf]["failed"], tf)
    fig.show()


## 5. What's next

| Notebook | Topic | Builds on this notebook |
|----------|-------|-----------------------|
| **NB05** | Legs & formation | Identifies the leg-in and leg-out around each valid base cluster → labels the formation (RBR / DBD / RBD / DBR) |
| **NB06** | Proximal / distal / departure | Calculates the exact zone price boundaries and departure strength |
| **NB07** | Verify all scenarios | End-to-end sanity check on synthetic fixtures A–G |

The `passed` list returned by `detect_bases()` feeds directly into NB05 — each entry defines a candidate zone whose formation still needs to be confirmed.